# Document Augmentation for Enhanced RAG Retrieval

## Overview

This notebook tackles one of the most persistent problems in Retrieval-Augmented Generation: **vocabulary mismatch**. When users ask questions, they rarely use the same words the source documents contain. A patient searches for "heart attack" while the medical literature says "myocardial infarction." A developer asks about "killing a process" while the documentation says "terminate the application."

Traditional RAG systems embed documents as-is and hope that semantic similarity is enough to bridge this gap. Often, it isn't. Document augmentation is an **indexing-time strategy** that enriches each chunk with additional representations — generated questions, summaries, keywords, and rephrasings — so that retrieval can match on *any* of these representations while still returning the original content.

## What We'll Build

1. **Understand the vocabulary mismatch problem** with concrete, measurable examples
2. **Generate questions** each chunk could answer, and index them alongside the chunk
3. **Create summaries** that capture the essence of each chunk in a single sentence
4. **Extract keywords** and entities for metadata-enriched retrieval
5. **Build a multi-representation index** that stores original text, questions, summaries, and keywords
6. **Compare retrieval quality** with and without augmentation
7. **Analyze the tradeoffs** of indexing cost vs retrieval quality

## Technical Stack

| Component | Choice |
|-----------|--------|
| LLM | `Qwen/Qwen3-14B` (4-bit NF4 quantization) |
| Embeddings | `BAAI/bge-base-en-v1.5` via sentence-transformers |
| Vector Store | FAISS (CPU) |
| Framework | Pure Python — no LangChain, no LlamaIndex |

In [ ]:
# --- Google Colab Setup: LLM + Embeddings ---
!pip install -q transformers>=4.51.0 accelerate bitsandbytes torch
!pip install -q sentence-transformers faiss-cpu numpy

import torch, json, re, time, textwrap
import numpy as np
from google.colab import drive
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from sentence_transformers import SentenceTransformer
import faiss

# --- Mount Google Drive for model caching ---
drive.mount('/content/drive')
CACHE_DIR = '/content/drive/MyDrive/models'

# --- Load Qwen3-14B with 4-bit quantization ---
MODEL_NAME = 'Qwen/Qwen3-14B'
quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_quant_type='nf4',
)
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, cache_dir=CACHE_DIR)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME, device_map='auto', quantization_config=quantization_config,
    cache_dir=CACHE_DIR, torch_dtype='auto',
)

# --- Load BGE embedding model ---
embed_model = SentenceTransformer('BAAI/bge-base-en-v1.5', cache_folder=CACHE_DIR)

def generate(messages, max_new_tokens=1024, temperature=0.6):
    """Generate a response from the LLM given chat messages."""
    text = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True, enable_thinking=False
    )
    inputs = tokenizer([text], return_tensors='pt').to(model.device)
    output_ids = model.generate(
        **inputs, max_new_tokens=max_new_tokens,
        temperature=temperature, do_sample=True, top_k=20,
    )
    generated = output_ids[0][inputs.input_ids.shape[1]:]
    return tokenizer.decode(generated, skip_special_tokens=True)

print(f'LLM loaded: {MODEL_NAME}')
print(f'Embedding model loaded: BAAI/bge-base-en-v1.5 (dim={embed_model.get_sentence_embedding_dimension()})')
print(f'Device: {model.device}')

## Part 1: The Vocabulary Mismatch Problem

### Why Semantic Search Isn't Enough

Embedding models map text into a shared vector space where semantically similar texts are nearby. This works well when the query and document share semantic overlap. But real-world queries and documents often **describe the same concept using entirely different vocabulary**:

| User Query | Document Text |
|---|---|
| "heart attack symptoms" | "clinical presentation of myocardial infarction" |
| "how to kill a process" | "methods for terminating application execution" |
| "cheap flights to Paris" | "budget-friendly airfare options to Charles de Gaulle" |
| "fix slow computer" | "optimizing system performance and resource allocation" |

Even good embedding models struggle with these gaps because:

1. **Domain jargon vs. colloquial language**: Experts and laypersons describe the same phenomena differently
2. **Abstraction level mismatch**: Users ask specific questions; documents often state general principles
3. **Implicit vs. explicit information**: The answer may be *implied* by the document but never directly stated

Let's see this problem in action with a concrete knowledge base.

In [ ]:
# === Synthetic Medical Knowledge Base ===
# Deliberately written in formal, clinical language — the kind users DON'T use.

KNOWLEDGE_BASE = [
    {
        "id": "cardio-001",
        "title": "Acute Myocardial Infarction",
        "content": (
            "Acute myocardial infarction (AMI) results from the sudden occlusion of a coronary "
            "artery, typically due to thrombotic rupture of an atherosclerotic plaque. The "
            "pathophysiology involves ischemic necrosis of myocardial tissue distal to the "
            "obstruction. Clinical presentation includes substernal chest discomfort often "
            "described as pressure or squeezing, with potential radiation to the left arm, jaw, "
            "or epigastrium. Diaphoresis, dyspnea, and nausea frequently accompany the primary "
            "symptom. Diagnosis relies on serial troponin measurements, 12-lead electrocardiography "
            "showing ST-segment elevation or depression, and clinical assessment. Immediate "
            "management includes dual antiplatelet therapy, anticoagulation, and emergent "
            "percutaneous coronary intervention or fibrinolytic therapy within established "
            "time windows."
        ),
    },
    {
        "id": "neuro-001",
        "title": "Cerebrovascular Accident Pathophysiology",
        "content": (
            "A cerebrovascular accident (CVA), commonly classified as ischemic or hemorrhagic, "
            "represents an acute disruption of cerebral perfusion. Ischemic CVA accounts for "
            "approximately 87% of cases and results from thrombotic or embolic occlusion of "
            "cerebral vasculature. The resulting hypoperfusion leads to a central core of "
            "irreversible neuronal death surrounded by a potentially salvageable penumbral zone. "
            "Hemorrhagic CVA involves rupture of cerebral vessels with extravasation of blood "
            "into brain parenchyma or subarachnoid space. Clinical manifestations depend on the "
            "affected vascular territory and may include sudden-onset hemiparesis, aphasia, "
            "visual field deficits, ataxia, and altered consciousness. Time-critical intervention "
            "with intravenous thrombolysis or mechanical thrombectomy significantly improves "
            "outcomes in ischemic events when administered within the therapeutic window."
        ),
    },
    {
        "id": "endo-001",
        "title": "Type 2 Diabetes Mellitus Management",
        "content": (
            "Type 2 diabetes mellitus (T2DM) is a metabolic disorder characterized by peripheral "
            "insulin resistance and progressive beta-cell dysfunction leading to chronic "
            "hyperglycemia. Glycemic control is assessed via hemoglobin A1c (HbA1c), with a "
            "target of less than 7.0% for most adults. First-line pharmacotherapy consists of "
            "metformin, which reduces hepatic glucose production and improves insulin sensitivity. "
            "Second-line agents include sodium-glucose cotransporter-2 (SGLT2) inhibitors, "
            "glucagon-like peptide-1 (GLP-1) receptor agonists, dipeptidyl peptidase-4 (DPP-4) "
            "inhibitors, and sulfonylureas. Lifestyle modifications — structured exercise programs, "
            "medical nutrition therapy, and weight management — remain foundational to treatment. "
            "Complications include retinopathy, nephropathy, neuropathy, and accelerated "
            "atherosclerotic cardiovascular disease requiring systematic screening protocols."
        ),
    },
    {
        "id": "pulm-001",
        "title": "Chronic Obstructive Pulmonary Disease",
        "content": (
            "Chronic obstructive pulmonary disease (COPD) encompasses emphysema and chronic "
            "bronchitis, characterized by persistent airflow limitation that is not fully "
            "reversible. The primary etiological factor is prolonged exposure to inhaled "
            "noxious particles, predominantly from tobacco smoke. Pathological changes include "
            "destruction of alveolar walls (emphysema), mucus hypersecretion and chronic "
            "inflammation of airways (chronic bronchitis), and small airway fibrosis. Spirometric "
            "assessment demonstrating a post-bronchodilator FEV1/FVC ratio below 0.70 confirms "
            "the diagnosis. Pharmacological management centers on inhaled bronchodilators — "
            "long-acting beta-2 agonists (LABA) and long-acting muscarinic antagonists (LAMA) — "
            "with inhaled corticosteroids added for patients with frequent exacerbations. "
            "Pulmonary rehabilitation programs combining exercise training, education, and "
            "behavioral modification improve functional capacity and quality of life."
        ),
    },
    {
        "id": "psych-001",
        "title": "Major Depressive Disorder",
        "content": (
            "Major depressive disorder (MDD) is a prevalent psychiatric condition characterized "
            "by persistent depressed mood or anhedonia lasting at least two weeks, accompanied "
            "by neurovegetative symptoms including sleep disturbance, appetite changes, "
            "psychomotor retardation or agitation, fatigue, diminished concentration, feelings "
            "of worthlessness or excessive guilt, and recurrent thoughts of death. The "
            "monoamine hypothesis implicates deficiency of serotonergic and noradrenergic "
            "neurotransmission, though contemporary models emphasize neuroplasticity, "
            "neuroinflammation, and hypothalamic-pituitary-adrenal axis dysregulation. "
            "First-line treatment includes selective serotonin reuptake inhibitors (SSRIs) "
            "and cognitive behavioral therapy (CBT). Treatment-resistant cases may benefit "
            "from serotonin-norepinephrine reuptake inhibitors (SNRIs), augmentation strategies, "
            "electroconvulsive therapy, or emerging interventions such as ketamine-based "
            "therapies and transcranial magnetic stimulation."
        ),
    },
    {
        "id": "ortho-001",
        "title": "Osteoarthritis Pathogenesis and Treatment",
        "content": (
            "Osteoarthritis (OA) is a degenerative joint disease characterized by progressive "
            "cartilage degradation, subchondral bone remodeling, osteophyte formation, and "
            "synovial inflammation. Weight-bearing joints — particularly the knee, hip, and "
            "spine — are most commonly affected. Risk factors include advanced age, obesity, "
            "prior joint injury, genetic predisposition, and repetitive occupational stress. "
            "The pathogenesis involves an imbalance between anabolic and catabolic processes "
            "in cartilage homeostasis, with matrix metalloproteinases (MMPs) and aggrecanases "
            "mediating tissue destruction. Clinical management follows a stepwise approach: "
            "non-pharmacological interventions (exercise therapy, weight reduction, assistive "
            "devices), pharmacological options (acetaminophen, topical and oral NSAIDs, "
            "intra-articular corticosteroid or hyaluronic acid injections), and ultimately "
            "surgical intervention (joint arthroplasty) for end-stage disease."
        ),
    },
]

print(f'Knowledge base loaded: {len(KNOWLEDGE_BASE)} documents')
for doc in KNOWLEDGE_BASE:
    print(f"  [{doc['id']}] {doc['title']} ({len(doc['content'].split())} words)")

### Demonstrating the Mismatch

Let's define queries using **everyday language** — the way real patients, not doctors, would phrase their questions. Then we'll measure how well pure embedding similarity retrieves the correct document.

The critical insight: even though `bge-base-en-v1.5` is an excellent embedding model, it can't fully bridge the gap between "heart attack" and "acute myocardial infarction" because these are surface-level vocabulary differences, not deep semantic ones that embedding training data always captures.

In [ ]:
# === Build a naive FAISS index (no augmentation) ===

def build_faiss_index(texts):
    """Embed a list of texts and build a FAISS index."""
    embeddings = embed_model.encode(texts, normalize_embeddings=True)
    dim = embeddings.shape[1]
    index = faiss.IndexFlatIP(dim)  # Inner product = cosine sim for normalized vectors
    index.add(embeddings.astype(np.float32))
    return index, embeddings

def retrieve_top_k(index, query, texts, k=3):
    """Retrieve top-k documents for a query."""
    q_emb = embed_model.encode([query], normalize_embeddings=True).astype(np.float32)
    scores, indices = index.search(q_emb, k)
    results = []
    for rank, (idx, score) in enumerate(zip(indices[0], scores[0])):
        results.append({"rank": rank + 1, "index": int(idx), "score": float(score), "text": texts[idx]})
    return results

# Index only the original document content
original_texts = [doc["content"] for doc in KNOWLEDGE_BASE]
original_ids = [doc["id"] for doc in KNOWLEDGE_BASE]
naive_index, _ = build_faiss_index(original_texts)

# Colloquial queries mapped to expected document IDs
TEST_QUERIES = [
    {"query": "What are the signs of a heart attack?", "expected": "cardio-001"},
    {"query": "My grandma had a stroke - what happens in the brain?", "expected": "neuro-001"},
    {"query": "How do you treat high blood sugar from diabetes?", "expected": "endo-001"},
    {"query": "Why does smoking mess up your lungs over time?", "expected": "pulm-001"},
    {"query": "What helps someone who is really depressed and nothing works?", "expected": "psych-001"},
    {"query": "My knee hurts when I walk - is it arthritis?", "expected": "ortho-001"},
]

print("=" * 80)
print("BASELINE RETRIEVAL (No Augmentation)")
print("=" * 80)

baseline_hits = 0
baseline_results = []
for tq in TEST_QUERIES:
    results = retrieve_top_k(naive_index, tq["query"], original_texts, k=1)
    retrieved_id = original_ids[results[0]["index"]]
    hit = retrieved_id == tq["expected"]
    baseline_hits += int(hit)
    baseline_results.append({"query": tq["query"], "expected": tq["expected"],
                             "retrieved": retrieved_id, "score": results[0]["score"], "hit": hit})
    status = "HIT" if hit else "MISS"
    print(f'\n[{status}] Query: "{tq["query"]}"')
    print(f'  Expected: {tq["expected"]} | Retrieved: {retrieved_id} | Score: {results[0]["score"]:.4f}')

print(f'\n{"=" * 80}')
print(f"Baseline Accuracy: {baseline_hits}/{len(TEST_QUERIES)} ({baseline_hits/len(TEST_QUERIES)*100:.0f}%)")
print("=" * 80)

## Part 2: What is Document Augmentation?

Document augmentation is an **indexing-time** strategy: before we ever receive a query, we enrich each document chunk with additional representations that bridge vocabulary gaps.

```
+------------------------------------------------------------------+
|                    INDEXING PIPELINE                              |
|                                                                  |
|  Original Chunk --+--> Generate Questions --> Embed & Index      |
|                   +--> Generate Summary   --> Embed & Index      |
|                   +--> Extract Keywords   --> Store as Metadata  |
|                   +--> Original Text      --> Embed & Index      |
|                                                                  |
|  All representations point back to the ORIGINAL chunk            |
+------------------------------------------------------------------+

+------------------------------------------------------------------+
|                    RETRIEVAL PIPELINE                             |
|                                                                  |
|  User Query --> Embed --> Search ALL representations             |
|                          --> Match question? Return chunk         |
|                          --> Match summary?  Return chunk         |
|                          --> Match original? Return chunk         |
+------------------------------------------------------------------+
```

### Why This Works

The key insight: **an LLM is much better at bridging vocabulary gaps than an embedding model**. When the LLM generates questions from a document about "myocardial infarction," it naturally produces questions like "What are the symptoms of a heart attack?" because it *understands* these are the same thing. The embedding model then only needs to match "heart attack" to "heart attack" — a trivial task.

### The Three Augmentation Strategies

| Strategy | What It Produces | Why It Helps |
|---|---|---|
| **Question Generation** | Questions the chunk answers | Bridges the query-document gap directly |
| **Summary Generation** | A concise one-sentence summary | Captures the *gist* for broad queries |
| **Keyword Extraction** | Key terms and entities | Enables exact-match hybrid retrieval |

## Part 3: Question Generation

### The Core Idea

For each document chunk, we ask the LLM: *"What questions could a user ask that this chunk would answer?"*

This is powerful because:
- The LLM translates **expert terminology** into **colloquial language**
- It anticipates **multiple phrasings** of the same question
- It identifies **implicit information** the chunk contains

At query time, the user's natural question matches one of these generated questions with high similarity, and we return the original chunk.

### Design Decisions

- **How many questions per chunk?** 3-5 is a good balance. More questions give better coverage but increase index size and LLM cost.
- **What style of questions?** Colloquial, varied — mix short and long, technical and informal.
- **Should we embed questions separately or concatenate?** Separately — each question targets a different vocabulary space.

In [ ]:
# === Question Generation ===

def generate_questions(chunk_text, n_questions=5):
    """Use the LLM to generate questions a user might ask about this chunk."""
    messages = [
        {"role": "system", "content": (
            "You are a helpful assistant that generates search queries. Given a document, "
            "generate exactly the requested number of diverse questions that this document "
            "could answer. Use EVERYDAY LANGUAGE - imagine a non-expert searching Google. "
            "Mix short and long questions. Do NOT use technical jargon unless a layperson "
            "would use it. Return ONLY the questions, one per line, numbered."
        )},
        {"role": "user", "content": (
            f"Generate {n_questions} diverse questions that this document answers:\n\n"
            f"{chunk_text}\n\n"
            f"Questions (use everyday language, one per line, numbered 1-{n_questions}):"
        )},
    ]
    response = generate(messages, max_new_tokens=512, temperature=0.7)
    # Parse numbered questions from the response
    questions = []
    for line in response.strip().split("\n"):
        line = re.sub(r"^\d+[.)\s]+", "", line.strip())
        if line and len(line) > 10:
            questions.append(line)
    return questions[:n_questions]

# Demonstrate on the first document (heart attack / myocardial infarction)
demo_doc = KNOWLEDGE_BASE[0]
print(f"Document: {demo_doc['title']}")
print(f"Content (first 200 chars): {demo_doc['content'][:200]}...")
print(f"\nGenerated Questions:")
demo_questions = generate_questions(demo_doc["content"])
for i, q in enumerate(demo_questions, 1):
    print(f"  {i}. {q}")

In [ ]:
# === Generate questions for ALL documents ===

augmented_questions = {}  # doc_id -> list of questions

print("Generating questions for each document...\n")
for doc in KNOWLEDGE_BASE:
    questions = generate_questions(doc["content"], n_questions=5)
    augmented_questions[doc["id"]] = questions
    print(f"[{doc['id']}] {doc['title']}")
    for q in questions:
        print(f"    -> {q}")
    print()

total_q = sum(len(v) for v in augmented_questions.values())
print(f"Total generated questions: {total_q} across {len(KNOWLEDGE_BASE)} documents")

### Testing Question-Augmented Retrieval

Now we build a FAISS index that contains **both** the original documents and the generated questions. Each entry in the index maps back to its source document. When a user query matches a generated question better than the original text, we still return the correct document.

In [ ]:
# === Build question-augmented FAISS index ===

qa_texts = []        # All texts to embed (originals + questions)
qa_doc_ids = []      # Which document each text belongs to
qa_types = []        # 'original' or 'question'

for i, doc in enumerate(KNOWLEDGE_BASE):
    # Add original text
    qa_texts.append(doc["content"])
    qa_doc_ids.append(doc["id"])
    qa_types.append("original")
    # Add generated questions
    for q in augmented_questions[doc["id"]]:
        qa_texts.append(q)
        qa_doc_ids.append(doc["id"])
        qa_types.append("question")

qa_index, _ = build_faiss_index(qa_texts)

print(f"Question-augmented index: {len(qa_texts)} entries")
print(f"  - {qa_types.count('original')} original documents")
print(f"  - {qa_types.count('question')} generated questions")

# Test retrieval
print(f'\n{"=" * 80}')
print("QUESTION-AUGMENTED RETRIEVAL")
print("=" * 80)

qa_hits = 0
for tq in TEST_QUERIES:
    results = retrieve_top_k(qa_index, tq["query"], qa_texts, k=1)
    retrieved_doc_id = qa_doc_ids[results[0]["index"]]
    matched_type = qa_types[results[0]["index"]]
    hit = retrieved_doc_id == tq["expected"]
    qa_hits += int(hit)
    status = "HIT" if hit else "MISS"
    print(f'\n[{status}] Query: "{tq["query"]}"')
    print(f'  Expected: {tq["expected"]} | Retrieved: {retrieved_doc_id} | Score: {results[0]["score"]:.4f}')
    print(f"  Matched via: {matched_type}")
    if matched_type == "question":
        print(f"  Matched text: \"{results[0]['text'][:120]}...\"")

print(f'\n{"=" * 80}')
print(f"Question-Augmented Accuracy: {qa_hits}/{len(TEST_QUERIES)} ({qa_hits/len(TEST_QUERIES)*100:.0f}%)")
print(f"Baseline Accuracy:          {baseline_hits}/{len(TEST_QUERIES)} ({baseline_hits/len(TEST_QUERIES)*100:.0f}%)")
print("=" * 80)

## Part 4: Summary Augmentation

### Why Summaries Help

Questions bridge the gap for *specific* queries, but some queries are **broad and exploratory**:
- "Tell me about diabetes management"
- "What are the treatment options for depression?"
- "Overview of lung disease from smoking"

These broad queries don't match specific questions well. They match **summaries** — concise descriptions of what the chunk is *about*.

### The Strategy

For each chunk, generate a **one-sentence summary** that captures its main topic and key points. This summary:
- Uses **simpler vocabulary** than the original (the LLM naturally simplifies)
- Is **shorter**, so its embedding is less diluted by irrelevant details
- Captures the **gist**, making it ideal for broad, topical queries

We embed the summary and store it in the index, pointing back to the original chunk. At retrieval time, a broad query matches the summary, but we return the full, detailed original.

In [ ]:
# === Summary Generation ===

def generate_summary(chunk_text):
    """Generate a one-sentence plain-language summary of a document chunk."""
    messages = [
        {"role": "system", "content": (
            "You are a helpful assistant that writes concise summaries. Given a medical "
            "document, write ONE sentence summarizing its main topic and key points. "
            "Use plain, everyday language - as if explaining to a friend. "
            "Do NOT use technical abbreviations or jargon. Return ONLY the summary sentence."
        )},
        {"role": "user", "content": f"Summarize this in one plain-language sentence:\n\n{chunk_text}"},
    ]
    response = generate(messages, max_new_tokens=150, temperature=0.5)
    # Take only the first sentence
    summary = response.strip().split("\n")[0].strip()
    return summary

# Generate summaries for all documents
augmented_summaries = {}  # doc_id -> summary

print("Generating summaries...\n")
for doc in KNOWLEDGE_BASE:
    summary = generate_summary(doc["content"])
    augmented_summaries[doc["id"]] = summary
    print(f"[{doc['id']}] {doc['title']}")
    print(f"    Summary: {summary}")
    print()

print(f"Generated {len(augmented_summaries)} summaries.")

In [ ]:
# === Build summary-augmented FAISS index ===

sa_texts = []      # Texts to embed
sa_doc_ids = []    # Which document each belongs to
sa_types = []      # 'original' or 'summary'

for doc in KNOWLEDGE_BASE:
    # Original
    sa_texts.append(doc["content"])
    sa_doc_ids.append(doc["id"])
    sa_types.append("original")
    # Summary
    sa_texts.append(augmented_summaries[doc["id"]])
    sa_doc_ids.append(doc["id"])
    sa_types.append("summary")

sa_index, _ = build_faiss_index(sa_texts)

# Test with BROAD queries that summaries should handle well
BROAD_QUERIES = [
    {"query": "Tell me about heart attack treatment", "expected": "cardio-001"},
    {"query": "Overview of stroke and what happens in the brain", "expected": "neuro-001"},
    {"query": "How is diabetes managed with medication?", "expected": "endo-001"},
    {"query": "Lung disease from smoking", "expected": "pulm-001"},
    {"query": "Treatment options for severe depression", "expected": "psych-001"},
    {"query": "Arthritis in the knees", "expected": "ortho-001"},
]

print("=" * 80)
print("SUMMARY-AUGMENTED RETRIEVAL (Broad Queries)")
print("=" * 80)

sa_hits = 0
for tq in BROAD_QUERIES:
    results = retrieve_top_k(sa_index, tq["query"], sa_texts, k=1)
    retrieved_doc_id = sa_doc_ids[results[0]["index"]]
    matched_type = sa_types[results[0]["index"]]
    hit = retrieved_doc_id == tq["expected"]
    sa_hits += int(hit)
    status = "HIT" if hit else "MISS"
    print(f'\n[{status}] Query: "{tq["query"]}"')
    print(f'  Expected: {tq["expected"]} | Retrieved: {retrieved_doc_id} | Score: {results[0]["score"]:.4f} | Via: {matched_type}')

print(f"\nSummary-Augmented Accuracy (Broad Queries): {sa_hits}/{len(BROAD_QUERIES)} ({sa_hits/len(BROAD_QUERIES)*100:.0f}%)")

## Part 5: Keyword Extraction

### Why Keywords Matter

Embeddings excel at **semantic similarity** but can miss **exact term matches**. If a user searches for "metformin," an embedding model might rank a document about diabetes management lower than one about drug metabolism simply because the embedding space doesn't perfectly preserve individual term importance.

Keyword extraction complements embeddings by:
1. **Identifying named entities**: drug names, disease names, anatomical terms
2. **Extracting key concepts**: treatment approaches, diagnostic methods
3. **Enabling hybrid retrieval**: combine embedding similarity with keyword overlap

### How We Use Keywords

Keywords serve as **metadata** rather than embedded representations. We store them alongside each chunk and can use them for:
- **Keyword boosting**: if a query term exactly matches a keyword, boost that chunk's retrieval score
- **Filtering**: narrow the search space before doing semantic search
- **Hybrid scoring**: combine cosine similarity with keyword overlap score

In [ ]:
# === Keyword Extraction ===

def extract_keywords(chunk_text, n_keywords=10):
    """Extract key terms, entities, and concepts from a chunk."""
    messages = [
        {"role": "system", "content": (
            "You are a helpful assistant that extracts keywords from documents. "
            "Given a document, extract the most important terms, including: "
            "disease names (both technical AND common names), drug names, symptoms, "
            "treatment methods, and key medical concepts. Include BOTH the technical "
            "term and the everyday equivalent when applicable. "
            "Return ONLY a comma-separated list of keywords, nothing else."
        )},
        {"role": "user", "content": f"Extract {n_keywords} important keywords from this document:\n\n{chunk_text}"},
    ]
    response = generate(messages, max_new_tokens=256, temperature=0.3)
    # Parse comma-separated keywords
    keywords = [kw.strip().lower() for kw in response.strip().split(",") if kw.strip()]
    return keywords[:n_keywords]

# Extract keywords for all documents
augmented_keywords = {}  # doc_id -> list of keywords

print("Extracting keywords...\n")
for doc in KNOWLEDGE_BASE:
    keywords = extract_keywords(doc["content"])
    augmented_keywords[doc["id"]] = keywords
    print(f"[{doc['id']}] {doc['title']}")
    print(f"    Keywords: {', '.join(keywords)}")
    print()

print(f"Extracted keywords for {len(augmented_keywords)} documents.")

In [ ]:
# === Hybrid Retrieval: Embedding + Keyword Scoring ===

def keyword_score(query, keywords):
    """Compute a simple keyword overlap score between query and document keywords."""
    query_terms = set(query.lower().split())
    keyword_set = set(keywords)
    # Check for full keyword matches (multi-word keywords)
    query_lower = query.lower()
    full_matches = sum(1 for kw in keyword_set if kw in query_lower)
    # Check for individual term overlap
    term_matches = len(query_terms & keyword_set)
    return full_matches * 2.0 + term_matches  # Weight full matches higher

def hybrid_retrieve(index, query, texts, doc_ids, keywords_map, k=1, alpha=0.7):
    """Retrieve using a weighted combination of embedding similarity and keyword score."""
    q_emb = embed_model.encode([query], normalize_embeddings=True).astype(np.float32)
    # Get more candidates than needed, then re-rank
    n_candidates = min(len(texts), 20)
    scores, indices = index.search(q_emb, n_candidates)

    candidates = []
    for idx, emb_score in zip(indices[0], scores[0]):
        doc_id = doc_ids[idx]
        kw_score = keyword_score(query, keywords_map.get(doc_id, []))
        # Normalize keyword score to [0, 1] range
        max_possible_kw = max(1, len(keywords_map.get(doc_id, [])))
        kw_norm = min(kw_score / max_possible_kw, 1.0)
        combined = alpha * float(emb_score) + (1 - alpha) * kw_norm
        candidates.append({"index": int(idx), "doc_id": doc_id, "emb_score": float(emb_score),
                           "kw_score": kw_score, "combined": combined})

    # Sort by combined score
    candidates.sort(key=lambda x: x["combined"], reverse=True)
    # Deduplicate by doc_id, keep highest score
    seen = set()
    results = []
    for c in candidates:
        if c["doc_id"] not in seen:
            seen.add(c["doc_id"])
            results.append(c)
            if len(results) == k:
                break
    return results

# Test keyword-boosted retrieval
KEYWORD_QUERIES = [
    {"query": "What is metformin used for?", "expected": "endo-001"},
    {"query": "SSRI medication for depression", "expected": "psych-001"},
    {"query": "troponin levels during chest pain", "expected": "cardio-001"},
    {"query": "COPD and FEV1 lung test", "expected": "pulm-001"},
    {"query": "knee replacement for arthritis", "expected": "ortho-001"},
    {"query": "thrombolysis for stroke treatment", "expected": "neuro-001"},
]

print("=" * 80)
print("HYBRID RETRIEVAL (Embeddings + Keyword Boosting)")
print("=" * 80)

hybrid_hits = 0
for tq in KEYWORD_QUERIES:
    results = hybrid_retrieve(naive_index, tq["query"], original_texts,
                              original_ids, augmented_keywords, k=1)
    top = results[0]
    hit = top["doc_id"] == tq["expected"]
    hybrid_hits += int(hit)
    status = "HIT" if hit else "MISS"
    print(f'\n[{status}] Query: "{tq["query"]}"')
    print(f"  Expected: {tq['expected']} | Retrieved: {top['doc_id']}")
    print(f"  Scores: embedding={top['emb_score']:.4f}, keyword={top['kw_score']:.1f}, combined={top['combined']:.4f}")

print(f"\nHybrid Accuracy: {hybrid_hits}/{len(KEYWORD_QUERIES)} ({hybrid_hits/len(KEYWORD_QUERIES)*100:.0f}%)")

## Part 6: Multi-Representation Indexing

### The Full Picture

So far we've tested each augmentation strategy independently. The real power comes from **combining all representations** into a single index. This creates a multi-representation retrieval system where:

1. **Specific questions** match users' natural-language queries ("What are the signs of a heart attack?")
2. **Summaries** match broad, exploratory queries ("Tell me about heart disease treatment")
3. **Original text** catches users who use technical language ("myocardial infarction pathophysiology")
4. **Keywords** boost results when exact terms appear in the query

### The Document Store Pattern

We separate the **retrieval index** from the **document store**:

```
FAISS Index (what we search)           Document Store (what we return)
+----------------------------+         +-----------------------------+
| "heart attack symptoms?"  ---+       |  Original chunk: "Acute     |
| "what causes chest pain?" ---+       |  myocardial infarction..." |
| "heart disease overview"  ---+---->  |                             |
| "Acute myocardial..."     ---+       |  + Keywords, metadata       |
+----------------------------+         +-----------------------------+
```

This is sometimes called the **"parent document retriever"** pattern: we search on child representations but return the parent.

## Part 7: Implementation from Scratch

Now we build the complete document augmentation pipeline. This is the full system you'd deploy in production:

1. **Chunk** the knowledge base (already done — our documents are chunk-sized)
2. **Augment** each chunk with questions + summary + keywords
3. **Embed** all representations (original + questions + summaries)
4. **Index** everything in FAISS with mappings back to original chunks
5. **Retrieve** by searching all representations, returning original chunks

In [ ]:
# === Full Document Augmentation Pipeline ===

class AugmentedDocument:
    """A document chunk with all its augmented representations."""

    def __init__(self, doc_id, title, content, questions=None, summary=None, keywords=None):
        self.doc_id = doc_id
        self.title = title
        self.content = content
        self.questions = questions or []
        self.summary = summary or ""
        self.keywords = keywords or []

    def all_representations(self):
        """Return list of (text, type) tuples for all searchable representations."""
        reps = [(self.content, "original")]
        for q in self.questions:
            reps.append((q, "question"))
        if self.summary:
            reps.append((self.summary, "summary"))
        return reps

    def __repr__(self):
        return (f"AugmentedDocument(id={self.doc_id}, questions={len(self.questions)}, "
                f"keywords={len(self.keywords)}, summary={bool(self.summary)})")


class MultiRepresentationIndex:
    """FAISS index over multiple representations of documents."""

    def __init__(self, embed_model):
        self.embed_model = embed_model
        self.index = None
        self.entries = []  # List of {"text", "type", "doc_id", "doc"}

    def add_documents(self, augmented_docs):
        """Build the index from a list of AugmentedDocument objects."""
        texts = []
        for doc in augmented_docs:
            for text, rep_type in doc.all_representations():
                self.entries.append({
                    "text": text, "type": rep_type,
                    "doc_id": doc.doc_id, "doc": doc
                })
                texts.append(text)

        embeddings = self.embed_model.encode(texts, normalize_embeddings=True)
        dim = embeddings.shape[1]
        self.index = faiss.IndexFlatIP(dim)
        self.index.add(embeddings.astype(np.float32))
        return len(texts)

    def search(self, query, k=3, keyword_boost=True, alpha=0.7):
        """Search the multi-representation index with optional keyword boosting."""
        q_emb = self.embed_model.encode([query], normalize_embeddings=True).astype(np.float32)
        n = min(len(self.entries), k * 5)
        scores, indices = self.index.search(q_emb, n)

        candidates = []
        for idx, score in zip(indices[0], scores[0]):
            entry = self.entries[idx]
            combined = float(score)
            kw_bonus = 0.0
            if keyword_boost:
                kw_bonus = keyword_score(query, entry["doc"].keywords)
                max_kw = max(1, len(entry["doc"].keywords))
                combined = alpha * float(score) + (1 - alpha) * min(kw_bonus / max_kw, 1.0)
            candidates.append({
                "doc_id": entry["doc_id"], "doc": entry["doc"],
                "matched_text": entry["text"], "matched_type": entry["type"],
                "emb_score": float(score), "kw_bonus": kw_bonus, "combined": combined,
            })

        candidates.sort(key=lambda x: x["combined"], reverse=True)
        # Deduplicate by doc_id
        seen = set()
        results = []
        for c in candidates:
            if c["doc_id"] not in seen:
                seen.add(c["doc_id"])
                results.append(c)
                if len(results) == k:
                    break
        return results

print("Classes defined: AugmentedDocument, MultiRepresentationIndex")
print("Ready to build the full pipeline.")

In [ ]:
# === Build Full Augmented Index ===

# We already generated questions, summaries, and keywords above.
# Now assemble them into AugmentedDocument objects.

augmented_docs = []
for doc in KNOWLEDGE_BASE:
    aug_doc = AugmentedDocument(
        doc_id=doc["id"],
        title=doc["title"],
        content=doc["content"],
        questions=augmented_questions[doc["id"]],
        summary=augmented_summaries[doc["id"]],
        keywords=augmented_keywords[doc["id"]],
    )
    augmented_docs.append(aug_doc)
    print(aug_doc)

# Build the multi-representation index
mr_index = MultiRepresentationIndex(embed_model)
n_entries = mr_index.add_documents(augmented_docs)

print(f"\nMulti-representation index built: {n_entries} total entries")
print(f"  - {len(KNOWLEDGE_BASE)} original documents")
print(f"  - {sum(len(d.questions) for d in augmented_docs)} generated questions")
print(f"  - {sum(1 for d in augmented_docs if d.summary)} summaries")
print(f"  - {sum(len(d.keywords) for d in augmented_docs)} keywords (metadata, not embedded)")

## Part 8: Retrieval Comparison

Now for the moment of truth. We'll test three retrieval approaches side-by-side:

1. **Baseline** — embed only original documents, no augmentation
2. **Question-augmented** — original + generated questions
3. **Full augmentation** — original + questions + summaries + keyword boosting

We'll use a diverse set of queries that test different aspects:
- **Colloquial queries**: everyday language ("heart attack symptoms")
- **Broad queries**: topical exploration ("diabetes overview")
- **Technical queries**: expert terminology ("SSRI mechanism")
- **Edge cases**: ambiguous or cross-topic queries

In [ ]:
# === Comprehensive Test Suite ===

ALL_TEST_QUERIES = [
    # Colloquial (vocabulary mismatch)
    {"query": "What are the signs of a heart attack?", "expected": "cardio-001", "type": "colloquial"},
    {"query": "My grandma had a stroke - what happens in the brain?", "expected": "neuro-001", "type": "colloquial"},
    {"query": "Why does smoking mess up your lungs over time?", "expected": "pulm-001", "type": "colloquial"},
    {"query": "My knee hurts bad when I walk - could it be arthritis?", "expected": "ortho-001", "type": "colloquial"},
    # Broad / exploratory
    {"query": "Tell me about diabetes management", "expected": "endo-001", "type": "broad"},
    {"query": "Treatment options for depression", "expected": "psych-001", "type": "broad"},
    {"query": "Overview of heart disease and treatment", "expected": "cardio-001", "type": "broad"},
    {"query": "Chronic lung problems and how to treat them", "expected": "pulm-001", "type": "broad"},
    # Technical / precise
    {"query": "What is percutaneous coronary intervention?", "expected": "cardio-001", "type": "technical"},
    {"query": "SGLT2 inhibitors for glycemic control", "expected": "endo-001", "type": "technical"},
    {"query": "Mechanical thrombectomy indications", "expected": "neuro-001", "type": "technical"},
    {"query": "Matrix metalloproteinases in joint disease", "expected": "ortho-001", "type": "technical"},
]

print(f"Test suite: {len(ALL_TEST_QUERIES)} queries")
for qtype in ["colloquial", "broad", "technical"]:
    count = sum(1 for q in ALL_TEST_QUERIES if q["type"] == qtype)
    print(f"  {qtype}: {count} queries")

In [ ]:
# === Run all three retrieval methods side-by-side ===

def run_baseline(query):
    """Baseline: search only original documents."""
    results = retrieve_top_k(naive_index, query, original_texts, k=1)
    return original_ids[results[0]["index"]], results[0]["score"], "original"

def run_question_augmented(query):
    """Question-augmented: search original + questions."""
    results = retrieve_top_k(qa_index, query, qa_texts, k=1)
    return qa_doc_ids[results[0]["index"]], results[0]["score"], qa_types[results[0]["index"]]

def run_full_augmented(query):
    """Full augmentation: multi-representation + keyword boosting."""
    results = mr_index.search(query, k=1, keyword_boost=True)
    top = results[0]
    return top["doc_id"], top["combined"], top["matched_type"]

# Run comparison
print("=" * 100)
print(f"{"COMPREHENSIVE RETRIEVAL COMPARISON":^100}")
print("=" * 100)
print(f"{"Query":<55} {"Type":<12} {"Base":<6} {"Q-Aug":<6} {"Full":<6}")
print("-" * 100)

results_by_method = {"baseline": [], "question_aug": [], "full_aug": []}
results_by_type = {}

for tq in ALL_TEST_QUERIES:
    q = tq["query"]
    exp = tq["expected"]
    qtype = tq["type"]

    b_id, b_score, b_type = run_baseline(q)
    qa_id, qa_score, qa_type = run_question_augmented(q)
    f_id, f_score, f_type = run_full_augmented(q)

    b_hit = b_id == exp
    qa_hit = qa_id == exp
    f_hit = f_id == exp

    results_by_method["baseline"].append(b_hit)
    results_by_method["question_aug"].append(qa_hit)
    results_by_method["full_aug"].append(f_hit)

    if qtype not in results_by_type:
        results_by_type[qtype] = {"baseline": [], "question_aug": [], "full_aug": []}
    results_by_type[qtype]["baseline"].append(b_hit)
    results_by_type[qtype]["question_aug"].append(qa_hit)
    results_by_type[qtype]["full_aug"].append(f_hit)

    b_sym = "Y" if b_hit else "."
    qa_sym = "Y" if qa_hit else "."
    f_sym = "Y" if f_hit else "."

    short_q = q[:52] + "..." if len(q) > 55 else q
    print(f"{short_q:<55} {qtype:<12} {b_sym:<6} {qa_sym:<6} {f_sym:<6}")

print("=" * 100)

In [ ]:
# === Summary Statistics ===

print(f'\n{"=" * 70}')
print(f'{"RESULTS SUMMARY":^70}')
print(f'{"=" * 70}')

total = len(ALL_TEST_QUERIES)
for method_name, hits in results_by_method.items():
    acc = sum(hits) / total * 100
    print(f"  {method_name:<20}: {sum(hits)}/{total} correct ({acc:.0f}%)")

print(f'\n{"--- Accuracy by Query Type ---":^70}')
for qtype, methods in results_by_type.items():
    print(f"\n  {qtype.upper()}:")
    for method_name, hits in methods.items():
        n = len(hits)
        acc = sum(hits) / n * 100
        print(f"    {method_name:<20}: {sum(hits)}/{n} ({acc:.0f}%)")

print(f'\n{"=" * 70}')

### Understanding the Results

Let's analyze *why* each method succeeds or fails on different query types:

**Colloquial Queries** ("heart attack symptoms", "knee hurts"):
- **Baseline struggles** because the documents use clinical terminology ("myocardial infarction", "osteoarthritis")
- **Question augmentation excels** because the LLM generates questions using everyday language
- This is the scenario where document augmentation provides the biggest lift

**Broad Queries** ("diabetes management", "depression treatment"):
- **Summaries help** because they capture the general topic in plain language
- **Baseline may still work** if the query shares enough vocabulary with the document

**Technical Queries** ("percutaneous coronary intervention", "SGLT2 inhibitors"):
- **Baseline often works** because the query terms appear directly in the documents
- **Keywords provide a safety net** via exact-match boosting

The overall lesson: **no single strategy wins everywhere**. Multi-representation indexing provides robust coverage across all query types.

In [ ]:
# === Detailed Match Analysis: What did the full-augmented system match on? ===

print("=" * 90)
print(f'{"MATCH ANALYSIS: What representation was matched?":^90}')
print("=" * 90)

match_type_counts = {"original": 0, "question": 0, "summary": 0}

for tq in ALL_TEST_QUERIES:
    results = mr_index.search(tq["query"], k=1, keyword_boost=True)
    top = results[0]
    match_type_counts[top["matched_type"]] = match_type_counts.get(top["matched_type"], 0) + 1

    hit = top["doc_id"] == tq["expected"]
    status = "HIT" if hit else "MISS"
    print(f"\n[{status}] [{tq['type']:<11}] \"{tq['query']}\"")
    print(f"  Matched via [{top['matched_type']}]: \"{top['matched_text'][:100]}...\"")
    print(f"  Embedding: {top['emb_score']:.4f} | Keyword boost: {top['kw_bonus']:.1f} | Combined: {top['combined']:.4f}")

print(f'\n{"=" * 90}')
print("Match type distribution:")
for t, count in sorted(match_type_counts.items()):
    print(f"  {t}: {count} queries ({count/len(ALL_TEST_QUERIES)*100:.0f}%)")
print("=" * 90)

In [ ]:
# === Score Distribution Analysis ===

print("=" * 80)
print(f'{"SCORE ANALYSIS: How much does augmentation improve similarity?":^80}')
print("=" * 80)

for tq in ALL_TEST_QUERIES[:6]:  # Show first 6 for readability
    # Get baseline score for the correct document
    correct_idx = original_ids.index(tq["expected"])
    q_emb = embed_model.encode([tq["query"]], normalize_embeddings=True).astype(np.float32)
    doc_emb = embed_model.encode([original_texts[correct_idx]], normalize_embeddings=True).astype(np.float32)
    baseline_sim = float(np.dot(q_emb[0], doc_emb[0]))

    # Get best augmented score
    aug_results = mr_index.search(tq["query"], k=1, keyword_boost=False)
    aug_sim = aug_results[0]["emb_score"]
    aug_type = aug_results[0]["matched_type"]

    improvement = aug_sim - baseline_sim
    pct = (improvement / max(baseline_sim, 0.001)) * 100

    print(f"\nQuery: \"{tq['query'][:60]}...\"")
    print(f"  Baseline similarity:     {baseline_sim:.4f}")
    print(f"  Augmented similarity:    {aug_sim:.4f} (matched via {aug_type})")
    print(f"  Improvement:            {improvement:+.4f} ({pct:+.1f}%)")

print(f'\n{"=" * 80}')

## Part 9: Synthesis — Indexing Cost vs. Retrieval Quality

### The Cost of Augmentation

Document augmentation isn't free. For each chunk, we make **three LLM calls** at indexing time:
1. Generate questions (~512 output tokens)
2. Generate summary (~150 output tokens)
3. Extract keywords (~256 output tokens)

And we embed **more vectors** per chunk:
- Baseline: 1 embedding per chunk
- Augmented: 1 (original) + 5 (questions) + 1 (summary) = **7 embeddings per chunk**

### Cost Analysis

| Factor | Baseline | Augmented | Multiplier |
|--------|----------|-----------|------------|
| LLM calls at indexing | 0 | 3 per chunk | New cost |
| Embeddings to compute | 1 per chunk | ~7 per chunk | 7x |
| FAISS index size | N vectors | ~7N vectors | 7x |
| Retrieval latency | Same | Same* | 1x |
| Retrieval accuracy | Lower | Higher | Depends |

*FAISS search is O(N) for brute-force, but with IVF indexing the difference is negligible.

### When Is Augmentation Worth It?

**Augmentation makes sense when:**
- Your documents use **specialized vocabulary** that users don't share (medical, legal, academic)
- Users ask questions in **natural, colloquial language**
- **Retrieval accuracy directly impacts business value** (medical Q&A, legal research, customer support)
- Your knowledge base is **relatively static** (augmentation is a one-time cost per document)
- You can afford the **indexing-time LLM cost** (the query-time cost is unchanged)

**Augmentation may NOT be worth it when:**
- Documents and queries share similar vocabulary (e.g., developer docs for developers)
- The knowledge base changes **very frequently** (re-augmenting is expensive)
- You need **real-time indexing** of new documents
- Storage and embedding computation costs are a primary concern

In [ ]:
# === Quantify the Cost Tradeoff ===

n_docs = len(KNOWLEDGE_BASE)
n_questions_per_doc = 5

# Baseline costs
baseline_embeddings = n_docs
baseline_llm_calls = 0

# Augmented costs
aug_llm_calls = n_docs * 3  # questions + summary + keywords
aug_embeddings = n_docs * (1 + n_questions_per_doc + 1)  # original + questions + summary

# Retrieval results
baseline_acc = sum(results_by_method["baseline"]) / total * 100
full_acc = sum(results_by_method["full_aug"]) / total * 100

print("=" * 60)
print(f'{"COST-BENEFIT ANALYSIS":^60}')
print("=" * 60)
print(f"\nKnowledge base: {n_docs} documents")
print(f'\n{"Metric":<30} {"Baseline":<15} {"Augmented":<15}')
print("-" * 60)
print(f'{"LLM calls (indexing)":<30} {baseline_llm_calls:<15} {aug_llm_calls:<15}')
print(f'{"Embeddings computed":<30} {baseline_embeddings:<15} {aug_embeddings:<15}')
print(f'{"Index entries (vectors)":<30} {baseline_embeddings:<15} {aug_embeddings:<15}')
print(f'{"Retrieval accuracy":<30} {f"{baseline_acc:.0f}%":<15} {f"{full_acc:.0f}%":<15}')
print(f'{"Embedding multiplier":<30} {"1x":<15} {f"{aug_embeddings/baseline_embeddings:.0f}x":<15}')

print(f'\n{"=" * 60}')
print(f"Cost per document: 3 LLM calls + {1 + n_questions_per_doc + 1} embeddings")
print(f"Retrieval improvement: {full_acc - baseline_acc:+.0f} percentage points")

# At scale projection
for scale in [100, 1000, 10000]:
    llm_calls = scale * 3
    embeddings = scale * (1 + n_questions_per_doc + 1)
    print(f"\nAt {scale:,} documents:")
    print(f"  LLM calls needed: {llm_calls:,}")
    print(f"  Index entries:    {embeddings:,}")

print(f'\n{"=" * 60}')

### Alternative Augmentation Strategies

The techniques in this notebook are the foundation, but there are many variations:

| Strategy | Description | Best For |
|---|---|---|
| **HyDE** (Hypothetical Document Embeddings) | Generate a hypothetical answer to the query, embed THAT | Query-time augmentation (covered in another notebook) |
| **Chunk Expansion** | Include surrounding context from neighboring chunks | Long documents with implicit cross-references |
| **Multi-lingual Augmentation** | Translate chunks into languages your users might query in | International user bases |
| **Paraphrase Generation** | Generate rephrased versions of each chunk | Diverse writing styles in queries |
| **Entity Linking** | Link extracted entities to a knowledge graph | Domain-specific retrieval (medical, legal) |

### Document Augmentation vs. Query Augmentation

An important architectural distinction:

- **Document augmentation** (this notebook): enrich documents at **indexing time**. Cost is paid once per document. No latency impact at query time.
- **Query augmentation** (HyDE, query expansion): transform the query at **query time**. No indexing cost, but adds latency to every query.

In practice, many production systems use **both**: augmented documents in the index + query expansion at retrieval time. The two approaches are complementary, not competing.

### Production Implementation Notes

If you're deploying document augmentation in a real system, consider:

**1. Batching LLM calls**: Don't call the LLM once per chunk — batch multiple chunks into a single prompt ("Generate questions for each of these 5 chunks"). This reduces overhead and cost.

**2. Caching augmentations**: Store generated questions, summaries, and keywords alongside the original documents. When the document hasn't changed, don't regenerate.

**3. Quality filtering**: Not all generated questions are good. Filter by:
- Remove duplicates (cosine similarity > 0.95 between questions)
- Remove questions that are too generic ("What is this about?")
- Remove questions that aren't answerable from the chunk alone

**4. Incremental updates**: When a document changes, only re-augment the changed chunks. Use content hashing to detect changes.

**5. Monitoring**: Track which representation type (original, question, summary) is being matched most often. If questions dominate, consider generating more. If originals dominate, your documents may already be well-written for search.

### End-to-End: Retrieval + Generation

Let's close the loop by using the augmented retrieval system to answer a question. The pipeline:
1. User asks a question in everyday language
2. Multi-representation index retrieves the best chunk
3. LLM generates an answer using the retrieved chunk as context

In [ ]:
# === End-to-End RAG with Document Augmentation ===

def rag_answer(query, index, verbose=True):
    """Full RAG pipeline: retrieve + generate."""
    # Retrieve
    results = index.search(query, k=1, keyword_boost=True)
    top = results[0]
    context = top["doc"].content

    if verbose:
        print(f"Query: {query}")
        print(f"Retrieved: [{top['doc_id']}] {top['doc'].title}")
        print(f"Matched via: {top['matched_type']} (score: {top['combined']:.4f})")
        print(f"Context: {context[:150]}...")
        print()

    # Generate answer
    messages = [
        {"role": "system", "content": (
            "You are a helpful medical information assistant. Answer the user's question "
            "based ONLY on the provided context. Use simple, everyday language. "
            "If the context doesn't contain enough information, say so."
        )},
        {"role": "user", "content": f"Context:\n{context}\n\nQuestion: {query}"},
    ]
    answer = generate(messages, max_new_tokens=300, temperature=0.5)
    return answer, top

# Ask questions using everyday language
demo_questions = [
    "What are the warning signs that someone is having a heart attack?",
    "My doctor said I have high blood sugar - what medicines can help?",
    "What helps when antidepressants don't work for someone?",
]

for q in demo_questions:
    print("=" * 80)
    answer, top = rag_answer(q, mr_index)
    print(f"Answer:\n{answer}")
    print("=" * 80)
    print()

In [ ]:
# === Verify: Augmented retrieval returns the ORIGINAL content ===
# This is a critical check: we must return the original detailed document,
# not the generated question or summary that was matched.

print("=" * 80)
print("VERIFICATION: Augmented retrieval returns original content")
print("=" * 80)

query = "What are the warning signs of a heart attack?"
results = mr_index.search(query, k=1, keyword_boost=True)
top = results[0]

print(f'\nQuery: "{query}"')
print(f"\nMatched representation ({top['matched_type']}):")
print(f"  \"{top['matched_text'][:120]}...\"")
print(f"\nReturned content (original document):")
print(f"  \"{top['doc'].content[:200]}...\"")
print(f"\nDocument ID: {top['doc_id']}")
print(f"Document title: {top['doc'].title}")
print(f"Keywords: {', '.join(top['doc'].keywords[:5])}...")

# Confirm the returned content is the ORIGINAL, not the matched representation
is_original = top['doc'].content == next(
    d["content"] for d in KNOWLEDGE_BASE if d["id"] == top["doc_id"]
)
print(f"\nContent matches original document: {is_original}")
print("=" * 80)

In [ ]:
# === View All Augmentations for One Document ===

doc = augmented_docs[0]  # Heart attack document
print("=" * 80)
print(f"COMPLETE AUGMENTATION PROFILE: [{doc.doc_id}] {doc.title}")
print("=" * 80)

print(f"\n--- Original Content ({len(doc.content.split())} words) ---")
print(textwrap.fill(doc.content[:300], width=80) + "...")

print(f"\n--- Generated Questions ({len(doc.questions)}) ---")
for i, q in enumerate(doc.questions, 1):
    print(f"  {i}. {q}")

print(f"\n--- Summary ---")
print(f"  {doc.summary}")

print(f"\n--- Keywords ({len(doc.keywords)}) ---")
print(f"  {', '.join(doc.keywords)}")

print(f"\n--- Total Searchable Representations ---")
reps = doc.all_representations()
print(f"  {len(reps)} entries in the index for this single document")
for text, rtype in reps:
    preview = text[:80] + "..." if len(text) > 80 else text
    print(f"  [{rtype:<10}] {preview}")

print(f'\n{"=" * 80}')

## Key Takeaways

### 1. The Vocabulary Mismatch Problem Is Real and Measurable
Users and documents speak different languages. Embedding models narrow the gap but don't close it. We demonstrated this concretely: baseline retrieval fails when colloquial queries meet clinical documents.

### 2. Document Augmentation Bridges the Gap at Indexing Time
By generating questions, summaries, and keywords for each chunk, we create multiple "entry points" into the same document. The LLM does the vocabulary translation once, at indexing time, so every subsequent query benefits.

### 3. Different Augmentation Strategies Serve Different Query Types
- **Questions** help with specific, natural-language queries
- **Summaries** help with broad, exploratory queries
- **Keywords** help with exact-term matching and hybrid retrieval
- **Multi-representation indexing** combines all three for robust coverage

### 4. The Tradeoff Is Indexing Cost vs. Retrieval Quality
Augmentation multiplies your indexing cost (LLM calls + embeddings) by roughly 7x. But retrieval cost stays the same — FAISS search time is negligible. For knowledge bases with specialized vocabulary and non-expert users, this tradeoff is almost always worth it.

### 5. Augmentation Is Complementary to Other RAG Techniques
Document augmentation works alongside:
- **Query transformations** (HyDE, query expansion) — augment both sides
- **Reranking** — augmented retrieval feeds better candidates to the reranker
- **Contextual compression** — retrieve augmented, compress for the LLM
- **Semantic chunking** — better chunks produce better augmentations